Primary analyses for play and watch data reported in main paper

In [ ]:
"""Shared setup: imports, color/name lookups, game stimuli, human data."""

import json
import os
import random
import importlib
from collections import defaultdict
import sys
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr
from tqdm import tqdm

import analysis_utils
from analysis_utils import tidy_game_code
import constants

# ----- file/output paths -----
main_dir = constants.PROCESSED_RES_DIR
main_save = constants.FINAL_FIGURES_DIR
fig_data_file_pth = constants.FIG_DATA_DIR



import matplotlib as mpl 
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'


In [ ]:
# ----- game stimuli + human data -----
base_game_objs, idx2game, game2idx, game_stimuli = (
    analysis_utils.process_game_stimuli(constants.THINK_STIMULI_PTH)
)
human_df = pd.read_csv(constants.THINK_HUMAN_DATA)
all_game_types, game2game_type = analysis_utils.get_game_categories(human_df)


In [ ]:

model2name = dict(constants.PAPER_MODEL2NAME)
agent2color = dict(constants.PAPER_AGENT2COLOR)
game_type2fmt = constants.PAPER_GAME_TYPE2FMT
order_game_types = list(constants.PAPER_ORDER_GAME_TYPES)
model2color = {}


In [ ]:
''' 
Load play data
'''

viz_agents = ['ours', 'ours-depth3', 'expert', 'random']
with open(f"{main_dir}play/agg_scores_alpha_NEW.json", "r") as f: 
    fixed_alpha_res = json.load(f)

agg_scores_alpha= {}
individ_move_scores_alpha = {}
individ_move_scores_per_alpha_agent = {}


move_idx_scores_per_alpha_agent = {agent: {} for agent in viz_agents}
individ_move_scores_alpha_by_game = {agent: defaultdict(list) for agent in viz_agents}

individ_move_scores_per_alpha_per_game = {}


for agent_idx, agent in enumerate(viz_agents): 
    collapsed_scores = []
    alpha_vals = sorted(list(fixed_alpha_res[agent].keys()))
        
    individ_move_scores_alpha_by_game[agent] = defaultdict(list)
    
    individ_logprobs, game_metadata = fixed_alpha_res[agent][alpha_vals[0]][0] # [logprobs, games]
    print(len(individ_logprobs))
    n_matches = len(individ_logprobs) # take one to get shape for agg
    match_scores = np.zeros(n_matches)
    individ_move_scores = np.zeros_like(np.array(sum(individ_logprobs, [])))
    
    individ_move_scores_per_alpha= np.zeros([len(alpha_vals), len(np.array(sum(individ_logprobs, [])))])
    
    kept_alpha = set()
    
    
    
    for alpha_idx, alpha in enumerate(alpha_vals): 
        alpha_float = float(alpha)
        curr_move_idx = 0
        
        move_idx2game = {}
        
        individ_logprobs, game_metadata = fixed_alpha_res[agent][alpha][0]
        ran_games = set([game for game, _ in game_metadata])
        
        print(len(ran_games))
        
        game_list = []
        for game, arenas_in_game in game_metadata:
            if len(arenas_in_game) == 0: print("missing game: ", game) 
            game_list.extend([game for _ in range(len(arenas_in_game) * 2)]) # b/c per individ
        
        
        for match_idx in range(n_matches): 
            
            match_vals = individ_logprobs[match_idx]
            if np.any(['inf' in str(x) for x in match_vals]): print("inf " + match_vals)
            curr_game = game_list[match_idx]
                
            match_scores[match_idx] += sum(match_vals)
            for move_idx, move_val in enumerate(match_vals): 
                move_idx2game[curr_move_idx] = curr_game

                individ_move_scores[curr_move_idx] += move_val
                individ_move_scores_per_alpha[alpha_idx, curr_move_idx] = move_val
                curr_move_idx+= 1

                move_idx_scores_per_alpha_agent[agent].setdefault(alpha, {})
                move_idx_scores_per_alpha_agent[agent][alpha].setdefault(move_idx, [])
                move_idx_scores_per_alpha_agent[agent][alpha][move_idx].append(move_val)
                
            kept_alpha.add(alpha)
    match_scores /= len(kept_alpha)
    individ_move_scores /= len(kept_alpha)
    
    agg_scores_alpha[agent] = match_scores #collapsed_scores
    individ_move_scores_alpha[agent] = individ_move_scores
    
    
    individ_move_scores_per_alpha_per_game[agent] = {}
    for move_idx in range(curr_move_idx): 
        game = move_idx2game[move_idx]
        individ_move_scores_per_alpha_per_game[agent].setdefault(game, [])
        individ_move_scores_per_alpha_per_game[agent][game].append(individ_move_scores[move_idx])
    
    
    individ_move_scores_per_alpha_agent[agent] = individ_move_scores_per_alpha


match_counts = 0
for game, arenas in game_metadata: 
    match_counts += len(arenas) * 2
print(match_counts)

In [ ]:
agent2viz = {'ours': "Intuitive Gamer", 'expert': 'Expert Gamer', 'ours-depth3': 'Intuitive Gamer (Depth-3)'}


In [ ]:

fig, ax = plt.subplots(figsize=(5,4))
alpha_float_vals = [float(a) for a in alpha_vals]

n_boot = 100



for agent, res in individ_move_scores_per_alpha_agent.items():
    if agent == 'random':
        continue
    print(np.shape(res))
    # res is shape (n_alphas, n_replicates)
    
    means, lower, upper = analysis_utils.bootstrap_column_subsample(res)

    yerr = np.vstack([means - lower,
                      upper - means])

    ax.errorbar(
        alpha_float_vals, means,
        yerr=yerr,
        fmt='o',  
        capsize=5,      
        alpha=0.6,
        color = agent2color[agent]      
    )
    plt.scatter(alpha_float_vals, means, s=100, alpha=0.5,label=agent2viz[agent],color = agent2color[agent]   )

ax.set_xlabel("α",fontsize=18)
ax.set_ylabel("Average Move Logprob",fontsize=18)

plt.tight_layout()
# plt.show()


In [ ]:
n_boot = 1000
sub_vizagents = [a for a in viz_agents if 'depth3' not in a]

agent_stats = {}
for agent in sub_vizagents:
    # one mean per game, then bootstrap over those means
    per_game = [np.mean(v) for v in individ_move_scores_per_alpha_per_game[agent].values()]
    m, lo, hi = analysis_utils.bootstrap_mean_lo_hi(per_game, n_boot=n_boot)
    agent_stats[agent] = {'mean': m, 'lower': lo, 'upper': hi}

analysis_utils.plot_agent_bar_with_ci(
    agent_stats, sub_vizagents, agent2color, model2name,
    ylabel="Move Log Likelihood",
    savepath='final_figures/play/aggregate_logprobs_bar_bootstrap.svg',
    invert_y=True,
    csv_path=fig_data_file_pth + 'play_main.csv',
)
for agent in sub_vizagents:
    s = agent_stats[agent]
    print(f"{model2name[agent]}: mean={s['mean']:.4f}, "
          f"95% CI=[{s['lower']:.4f}, {s['upper']:.4f}]")


In [ ]:
n_boot = 1000
sub_vizagents = [a for a in viz_agents if 'depth3' not in a]

agent_stats = {}
for agent in sub_vizagents:
    m, lo, hi = analysis_utils.bootstrap_mean_lo_hi(
        agg_scores_alpha[agent], n_boot=n_boot
    )
    agent_stats[agent] = {'mean': m, 'lower': lo, 'upper': hi}

analysis_utils.plot_agent_bar_with_ci(
    agent_stats, sub_vizagents, agent2color, model2name,
    ylabel="Move Log Likelihood",
    savepath='final_figures/play/aggregate_logprobs_bar_bootstrap_agg_moves.svg',
    invert_y=True,
)
for agent in sub_vizagents:
    s = agent_stats[agent]
    print(f"{model2name[agent]}: mean={s['mean']:.4f}, "
          f"95% CI=[{s['lower']:.4f}, {s['upper']:.4f}]")


In [ ]:


agents = [a for a in viz_agents if 'depth3' not in a]
colors = [agent2color[agent] for agent in agents]

sorted_games = sorted(individ_move_scores_per_alpha_per_game[agents[0]].keys())
n_games = len(sorted_games)
x = np.arange(n_games)
width = 0.25 


means = np.zeros((n_games, len(agents)))
cis = np.zeros((n_games, len(agents), 2))

for agent_idx, agent in enumerate(agents):
    for game_idx, game in enumerate(sorted_games):
        scores = individ_move_scores_per_alpha_per_game[agent][game]
        means[game_idx, agent_idx] = np.mean(scores)
        boot_samples = np.random.choice(scores, (1000, len(scores)), replace=True).mean(axis=1)
        cis[game_idx, agent_idx, 0] = np.percentile(boot_samples, 2.5)
        cis[game_idx, agent_idx, 1] = np.percentile(boot_samples, 97.5)

sorted_order = np.argsort(means[:, 0])[::-1]
means_sorted = means[sorted_order, :]
cis_sorted = cis[sorted_order, :, :]

spacing = width * len(agents) 

plt.figure(figsize=(18, 6)) 

for i, agent in enumerate(agents):
    x_offset = x + i * width - spacing / 2
    yerr = [means_sorted[:, i] - cis_sorted[:, i, 0], cis_sorted[:, i, 1] - means_sorted[:, i]]
    plt.bar(x_offset, means_sorted[:, i], width,
            label=agent, color=colors[i], edgecolor='black', alpha=0.7, yerr=yerr, capsize=4)

plt.xticks(x, [tidy_game_code(sorted_games[idx]) for idx in sorted_order],
           rotation=45, ha='right', fontsize=16)
plt.ylabel("Move Likelihood", fontsize=24)
plt.tight_layout()
plt.gca().invert_yaxis()

plt.savefig('final_figures/play/aggregate_logprobs_per_game.pdf', dpi=400)
plt.show()



In [ ]:


# get diff of means
comparisons = [('ours', 'expert'), ('ours', 'random')]
comp_diffs=  {}
for comp in comparisons: 
    agent1, agent2 = comp 
    diffs = [a1 - a2 for a1, a2 in zip(individ_move_scores_alpha[agent1], individ_move_scores_alpha[agent2])]
    comp_diffs[tuple(comp)] = diffs
plt.hist(diffs)

In [ ]:


comparisons = [('ours', 'expert'), ('ours', 'random')]
comp_diffs = {}
comp_stats = {}

for comp in comparisons:
    agent1, agent2 = comp
    
    # Calculate differences
    diffs = [a1 - a2 for a1, a2 in zip(agg_scores_alpha[agent1], agg_scores_alpha[agent2])]
    comp_diffs[tuple(comp)] = diffs
    
    # Run one-sided paired t-test
    t_stat, p_value = stats.ttest_rel(agg_scores_alpha[agent1], agg_scores_alpha[agent2], alternative='greater')
    
    # Store results
    comp_stats[tuple(comp)] = {
        'mean_diff': np.mean(diffs),
        't_statistic': t_stat,
        'p_value': p_value,
        'significant': p_value < 0.05
    }

for comp, stats_dict in comp_stats.items():
    print(f"Comparison: {comp[0]} vs {comp[1]}")
    print(f"Mean difference: {stats_dict['mean_diff']:.4f}")
    print(f"t-statistic: {stats_dict['t_statistic']:.4f}")
    print(f"p-value: {stats_dict['p_value']}")#:.4f}")
    print(f"Statistically significant (p<0.05): {stats_dict['significant']}")
    print("-" * 40)

plt.figure(figsize=(12, 5))
for i, (comp, diffs) in enumerate(comp_diffs.items()):
    plt.subplot(1, len(comparisons), i+1)
    plt.hist(diffs, bins=10, alpha=0.7)
    plt.axvline(x=0, color='red', linestyle='--')
    plt.title(f"{comp[0]} vs {comp[1]}\nMean diff: {np.mean(diffs):.4f}")
    plt.xlabel("Difference (Group1 - Group2)")
    plt.ylabel("Frequency")
n = len(agg_scores_alpha[agent1])
print(f"t({n-1}) = {t_stat:.2f}")

plt.tight_layout()
plt.show()

In [ ]:


comparisons = [('ours', 'expert'), ('ours', 'random')]
comp_diffs = {}
comp_stats = {}

for comp in comparisons:
    agent1, agent2 = comp
    
    # Calculate differences
    diffs = [a1 - a2 for a1, a2 in zip(individ_move_scores_alpha[agent1], individ_move_scores_alpha[agent2])]
    comp_diffs[tuple(comp)] = diffs
    
    # Run one-sided paired t-test
    t_stat, p_value = stats.ttest_rel(individ_move_scores_alpha[agent1], individ_move_scores_alpha[agent2], alternative='greater')
    
    # Store results
    comp_stats[tuple(comp)] = {
        'mean_diff': np.mean(diffs),
        't_statistic': t_stat,
        'p_value': p_value,
        'significant': p_value < 0.05
    }

for comp, stats_dict in comp_stats.items():
    print(f"Comparison: {comp[0]} vs {comp[1]}")
    print(f"Mean difference: {stats_dict['mean_diff']:.4f}")
    print(f"t-statistic: {stats_dict['t_statistic']:.4f}")
    print(f"p-value: {stats_dict['p_value']}")#:.4f}")
    print(f"Statistically significant (p<0.05): {stats_dict['significant']}")
    print("-" * 40)

plt.figure(figsize=(12, 5))
for i, (comp, diffs) in enumerate(comp_diffs.items()):
    plt.subplot(1, len(comparisons), i+1)
    plt.hist(diffs, bins=10, alpha=0.7)
    plt.axvline(x=0, color='red', linestyle='--')
    plt.title(f"{comp[0]} vs {comp[1]}\nMean diff: {np.mean(diffs):.4f}")
    plt.xlabel("Difference (Group1 - Group2)")
    plt.ylabel("Frequency")
n = len(individ_move_scores_alpha[agent1])
print(f"t({n-1}) = {t_stat:.2f}")

plt.tight_layout()
plt.show()

In [ ]:
''' 
Load play data
'''

viz_agents = ['ours', 'ours-depth3', 'expert', 'random']
game_stages = ['early', 'middle', 'late']
with open(f"{main_dir}play/game_stages_alpha_NEW.json", "r") as f: 
    fixed_alpha_res_stages = json.load(f)
    
fig,axes = plt.subplots(1,len(game_stages), figsize=(16,4), sharey=True)
    
for game_stage_idx, game_stage in enumerate(game_stages): 
    ax = axes[game_stage_idx]
    fixed_alpha_res = fixed_alpha_res_stages[game_stage]

    agg_scores_alpha= {}
    individ_move_scores_alpha = {}
    individ_move_scores_per_alpha_agent = {}
    for agent_idx, agent in enumerate(viz_agents): 
        collapsed_scores = []
        alpha_vals = sorted(list(fixed_alpha_res[agent].keys()))

        individ_logprobs, game_metadata = fixed_alpha_res[agent][alpha_vals[0]][0] # [logprobs, games]
        print(len(individ_logprobs))
        n_matches = len(individ_logprobs) # take one to get shape for agg
        match_scores = np.zeros(n_matches)
        individ_move_scores = np.zeros_like(np.array(sum(individ_logprobs, [])))
        
        individ_move_scores_per_alpha= np.zeros([len(alpha_vals), len(np.array(sum(individ_logprobs, [])))])
        
        kept_alpha = set()
        for alpha_idx, alpha in enumerate(alpha_vals): 
            alpha_float = float(alpha)
            curr_move_idx = 0
            for match_idx in range(n_matches): 
                
                individ_logprobs, game_metadata = fixed_alpha_res[agent][alpha][0]
                match_vals = individ_logprobs[match_idx]
                if np.any(['inf' in str(x) for x in match_vals]): print("inf " + match_vals)
                        
                match_scores[match_idx] += sum(match_vals)
                for move_idx, move_val in enumerate(match_vals): 
                    
                    individ_move_scores[curr_move_idx] += move_val
                    individ_move_scores_per_alpha[alpha_idx, curr_move_idx] = move_val
                    curr_move_idx+= 1
                kept_alpha.add(alpha)
        match_scores /= len(kept_alpha)
        individ_move_scores /= len(kept_alpha)
        agg_scores_alpha[agent] = match_scores #collapsed_scores
        individ_move_scores_alpha[agent] = individ_move_scores
        individ_move_scores_per_alpha_agent[agent] = individ_move_scores_per_alpha

    n_boot = 1000
    agent_stats = {}
    for agent in viz_agents:
        vals = individ_move_scores_alpha[agent]
        mean, lower, upper = analysis_utils.bootstrap_column_subsample(vals, n_boot=n_boot) 
        agent_stats[agent] = {
            'mean': mean[0],
            'lower': lower[0],
            'upper': upper[0],
        }

    agents = [model2name[agent].split(" (Partial)")[0].replace(" ", "\n") for agent in viz_agents]
    means = [agent_stats[agent]['mean'] for agent in viz_agents]
    colors = [agent2color[agent] for agent in viz_agents]

    ax.bar(agents, means, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

    yerr = [
        [agent_stats[agent]['mean'] - agent_stats[agent]['lower'] for agent in viz_agents],  # lower errors
        [agent_stats[agent]['upper'] - agent_stats[agent]['mean'] for agent in viz_agents]   # upper errors
    ]
    ax.errorbar(agents, means, yerr=yerr, fmt='none', ecolor='black', capsize=5, linewidth=1.5)

    ax.set_xlabel("", fontsize=18)
    ax.set_ylabel("Move Log Likelihood", fontsize=18)
    ax.tick_params(axis='x', labelsize=16)
    ax.invert_yaxis()  
    ax.set_title(game_stage.capitalize(), fontsize=18)
    for agent in viz_agents:
        print(f"{game_stage.capitalize()} {model2name[agent]}: mean={agent_stats[agent]['mean']:.4f}, 95% CI=[{agent_stats[agent]['lower']:.4f}, {agent_stats[agent]['upper']:.4f}]")
 
plt.tight_layout()           
plt.savefig('final_figures/play/aggregate_logprobs_bar_bootstrap_stages.pdf', dpi=400)


In [ ]:
viz_agents = ['ours', 'ours-depth3', 'expert', 'random']

with open(main_dir + "play/per_game_admixture.json") as f: 
    per_game_admixture = json.load(f)
    
with open(main_dir + "play/per_participant_admixture.json") as f: 
    participant_bootstrap_res_temp = json.load(f)


In [ ]:
temp_step = 0.5
temp_vals = [str(temp) for temp in np.arange(0.5, 3.0 + temp_step, temp_step)]
agent_props_participant = {agent: [] for agent in viz_agents}
agent2viz['random'] = 'Random Gamer'
temp2scores = {temp: [] for temp in temp_vals}

temp2agent_scores = {agent: {temp: [] for temp in temp_vals} for agent in viz_agents}

for participant, entries in per_game_admixture.items():
    agent_vals = {}
    for temp, entry in entries.items(): 
        (props, score) = entry[0] # b/c just one sample
        if str(score) == 'nan': continue
        for agent, val in props.items(): 
            agent_vals.setdefault(agent, [])
            agent_vals[agent].append(val)
        temp2scores[temp].append(score)
        for agent in props: 
            temp2agent_scores[agent][temp].append(props[agent])
    for agent, vals in agent_vals.items(): agent_props_participant[agent].append(np.mean(vals))

fig, axes = plt.subplots(1, 2, figsize=(10,4))

# -- Plot 1: Aggregate Log Probabilities with CI
ax = axes[0]
means, lower, upper = [], [], []
n_boot = 1000
for temp in temp_vals:
    score_data = np.array(temp2scores[str(temp)])
    print("score data len: ", len(score_data))
    sub_means = []
    for _ in range(n_boot): 
        subvals = np.random.choice(score_data, len(score_data), replace=True)
        sub_means.append(np.mean(subvals))
    
    means.append(np.mean(sub_means))
    lower.append(np.percentile(sub_means, 2.5))
    upper.append(np.percentile(sub_means, 97.5))
    
yerr = np.vstack([np.array(means) - np.array(lower), np.array(upper) - np.array(means)])
#yerr = np.vstack([np.array(lower), np.array(upper)])
temp_vals = [float(t) for t in temp_vals]
ax.errorbar(temp_vals, means, yerr=yerr, fmt='o', capsize=4, alpha=0.9, color='black')
ax.set_xlabel("Temperature", fontsize=18)
ax.set_ylabel("Aggregate Log Prob", fontsize=18)

# -- Plot 2: Mixture Weights with CI
ax = axes[1]
for agent in viz_agents:
    means, lower, upper = [], [], []
    for temp in temp_vals:
        weight_data = temp2agent_scores[agent][str(temp)]
        
        print("len: ", agent, temp, len(weight_data))
        sub_means = []
        for _ in range(n_boot): 
            subvals = np.random.choice(weight_data, len(weight_data), replace=True)
            sub_means.append(np.mean(subvals))

        means.append(np.mean(weight_data))
        lower.append(np.std(weight_data))
        upper.append(np.std(weight_data))
        
    yerr =np.vstack([np.array(lower), np.array(upper)])
    ax.errorbar(temp_vals, means, yerr=yerr, 
                fmt='o', capsize=5, alpha=0.6,
                color=agent2color[agent], label=agent2viz[agent])
    ax.scatter(temp_vals, means, s=100, alpha=0.5, color=agent2color[agent])

ax.set_xlabel("Temperature", fontsize=18)
ax.set_ylabel("Mixture Weight", fontsize=18)
# ax.legend()
plt.tight_layout()

In [ ]:
''' 
Load play data
'''

viz_agents = ['ours', 'ours-depth3', 'expert', 'random']
game_stages = ['early', 'middle', 'late']
with open(f"{main_dir}play/game_stages_alpha.json", "r") as f: 
    fixed_alpha_res_stages = json.load(f)
    
fig,axes = plt.subplots(1,len(game_stages), figsize=(16,4), sharey=True)
    
for game_stage_idx, game_stage in enumerate(game_stages): 
    ax = axes[game_stage_idx]
    fixed_alpha_res = fixed_alpha_res_stages[game_stage]

    agg_scores_alpha= {}
    individ_move_scores_alpha = {}
    individ_move_scores_per_alpha_agent = {}
    for agent_idx, agent in enumerate(viz_agents): 
        collapsed_scores = []
        alpha_vals = sorted(list(fixed_alpha_res[agent].keys()))
        
        individ_logprobs, game_metadata = fixed_alpha_res[agent][alpha_vals[0]][0] # [logprobs, games]
        print(len(individ_logprobs))
        n_matches = len(individ_logprobs) # take one to get shape for agg
        match_scores = np.zeros(n_matches)
        individ_move_scores = np.zeros_like(np.array(sum(individ_logprobs, [])))
        
        individ_move_scores_per_alpha= np.zeros([len(alpha_vals), len(np.array(sum(individ_logprobs, [])))])
        
        kept_alpha = set()
        for alpha_idx, alpha in enumerate(alpha_vals): 
            alpha_float = float(alpha)
            curr_move_idx = 0
            for match_idx in range(n_matches): 
                
                individ_logprobs, game_metadata = fixed_alpha_res[agent][alpha][0]

                match_vals = individ_logprobs[match_idx]
                if np.any(['inf' in str(x) for x in match_vals]): print("inf " + match_vals)
                        
                match_scores[match_idx] += sum(match_vals)
                for move_idx, move_val in enumerate(match_vals): 
                    
                    individ_move_scores[curr_move_idx] += move_val
                    individ_move_scores_per_alpha[alpha_idx, curr_move_idx] = move_val
                    curr_move_idx+= 1
                kept_alpha.add(alpha)
        match_scores /= len(kept_alpha)
        individ_move_scores /= len(kept_alpha)
        agg_scores_alpha[agent] = match_scores #collapsed_scores
        individ_move_scores_alpha[agent] = individ_move_scores
        individ_move_scores_per_alpha_agent[agent] = individ_move_scores_per_alpha

    # Number of bootstrap iterations
    n_boot = 1000

    # Calculate bootstrapped statistics for each agent
    agent_stats = {}
    for agent in viz_agents:
        vals = individ_move_scores_alpha[agent]
        mean, lower, upper = analysis_utils.bootstrap_column_subsample(vals, n_boot=n_boot) 
        agent_stats[agent] = {
            'mean': mean[0],
            'lower': lower[0],
            'upper': upper[0],
        }

    agents = [model2name[agent].split(" (Partial)")[0].replace(" ", "\n") for agent in viz_agents]
    means = [agent_stats[agent]['mean'] for agent in viz_agents]
    colors = [agent2color[agent] for agent in viz_agents]

    ax.bar(agents, means, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

    yerr = [
        [agent_stats[agent]['mean'] - agent_stats[agent]['lower'] for agent in viz_agents],  # lower errors
        [agent_stats[agent]['upper'] - agent_stats[agent]['mean'] for agent in viz_agents]   # upper errors
    ]
    ax.errorbar(agents, means, yerr=yerr, fmt='none', ecolor='black', capsize=5, linewidth=1.5)

    ax.set_xlabel("", fontsize=18)
    ax.set_ylabel("Move Log Likelihood", fontsize=18)
    ax.tick_params(axis='x', labelsize=16)
    ax.invert_yaxis()  
    ax.set_title(game_stage.capitalize(), fontsize=18)

    for agent in viz_agents:
        print(f"{game_stage.capitalize()} {model2name[agent]}: mean={agent_stats[agent]['mean']:.4f}, 95% CI=[{agent_stats[agent]['lower']:.4f}, {agent_stats[agent]['upper']:.4f}]")
 
plt.tight_layout()           
plt.savefig('final_figures/play/aggregate_logprobs_bar_bootstrap_stages.pdf', dpi=400)


In [ ]:

importlib.reload(analysis_utils)
agent2color['expert'] = 'blue'
agent2color['ours'] = 'red'
per_game_df, fig, ordered_game_ids = analysis_utils.plot_individ_bars(per_game_admixture, [agent for agent in viz_agents if agent != 'ours-depth3'],
                                                    filetag="bootstrap_game_individ_weights.svg", 
                                                    label_x=[True, 'Games'], main_save=main_save,model2name=model2name, agent2color=agent2color)
per_game_df.to_csv(fig_data_file_pth + 'admixture_games_played.csv')
participant_mres_df, fig, _ = analysis_utils.plot_individ_bars(participant_bootstrap_res_temp, [agent for agent in viz_agents if agent != 'ours-depth3'],
                                                            filetag="bootstrap_participants_individ_weights.svg", main_save=main_save,model2name=model2name, agent2color=agent2color)
per_game_df.to_csv(fig_data_file_pth + 'admixture_play_individ.csv')


In [ ]:

importlib.reload(analysis_utils)
with open(main_dir + "watch/per_game_admixture_watch.json") as f: 
    per_game_admixture_watch = json.load(f)
    
with open(main_dir + "watch/per_uid_admixture_watch.json") as f: 
    participant_bootstrap_res_temp_watch = json.load(f)

per_game_df, fig, _ = analysis_utils.plot_individ_bars(per_game_admixture_watch, [agent for agent in viz_agents if agent != 'ours-depth3'],
                                                    filetag="bootstrap_game_individ_weights_watch.svg", 
                                                    label_x=[True, 'Games'], main_save=main_save,model2name=model2name, agent2color=agent2color,
                                                    )#ordered_game_ids=ordered_game_ids)
per_game_df.to_csv(fig_data_file_pth + 'admixture_games_watched.csv')
participant_mres_df, fig, _= analysis_utils.plot_individ_bars(participant_bootstrap_res_temp_watch, [agent for agent in viz_agents if agent != 'ours-depth3'],
                                                            filetag="bootstrap_participants_individ_weights_watch.svg", main_save=main_save,model2name=model2name, agent2color=agent2color)

participant_mres_df.to_csv(fig_data_file_pth + 'admixture_watch_individ.csv')


In [ ]:
temp_step = 0.5
temp_vals = [str(temp) for temp in np.arange(0.5, 3.0 + temp_step, temp_step)]
agent_props_participant = {agent: [] for agent in viz_agents}
agent2viz['random'] = 'Random'
temp2scores = {temp: [] for temp in temp_vals}

temp2agent_scores = {agent: {temp: [] for temp in temp_vals} for agent in viz_agents}

for participant, entries in per_game_admixture_watch.items():
    agent_vals = {}
    for temp, entry in entries.items(): 
        (props, score) = entry[0] # b/c just one sample now
        if str(score) == 'nan': continue
        for agent, val in props.items(): 
            agent_vals.setdefault(agent, [])
            agent_vals[agent].append(val)
        temp2scores[temp].append(score)
        for agent in props: 
            temp2agent_scores[agent][temp].append(props[agent])
    for agent, vals in agent_vals.items(): agent_props_participant[agent].append(np.mean(vals))


fig, axes = plt.subplots(1, 2, figsize=(10,4))

ax = axes[0]
means, lower, upper = [], [], []
n_boot = 1000
for temp in temp_vals:
    score_data = np.array(temp2scores[str(temp)])
    print("score data len: ", len(score_data))
    sub_means = []
    for _ in range(n_boot): 
        subvals = np.random.choice(score_data, len(score_data), replace=True)
        sub_means.append(np.mean(subvals))
    
    means.append(np.mean(sub_means))
    lower.append(np.percentile(sub_means, 2.5))
    upper.append(np.percentile(sub_means, 97.5))
    
yerr = np.vstack([np.array(means) - np.array(lower), np.array(upper) - np.array(means)])
temp_vals = [float(t) for t in temp_vals]
ax.errorbar(temp_vals, means, yerr=yerr, fmt='o', capsize=4, alpha=0.9, color='black')
ax.set_xlabel("Temperature", fontsize=18)
ax.set_ylabel("Jensen-Shannon\nDivergence", fontsize=18)
''''''
ax = axes[1]
for agent in viz_agents:
    means, lower, upper = [], [], []
    for temp in temp_vals:
        weight_data = temp2agent_scores[agent][str(temp)]

        print("len: ", agent, temp, len(weight_data))
        sub_means = []
        for _ in range(n_boot): 
            subvals = np.random.choice(weight_data, len(weight_data), replace=True)
            sub_means.append(np.mean(subvals))

        means.append(np.mean(weight_data))
        lower.append(np.std(weight_data))
        upper.append(np.std(weight_data))
        
    yerr =np.vstack([np.array(lower), np.array(upper)])
    ax.errorbar(temp_vals, means, yerr=yerr, 
                fmt='o', capsize=5, alpha=0.6,
                color=agent2color[agent], label=agent2viz[agent])
    ax.scatter(temp_vals, means, s=100, alpha=0.5, color=agent2color[agent])

ax.set_xlabel("Temperature", fontsize=18)
ax.set_ylabel("Mixture Weight", fontsize=18)
plt.tight_layout()

In [ ]:
'''
Load watch data
'''

viz_agents = ['ours', 'ours-depth3', 'expert', 'random']
# metric_df = pd.read_csv(f"{main_dir}watch/tvd_agg.csv")
metric_df = pd.read_csv(f"{main_dir}watch/metrics_agg.csv")

metric2viz = {'TV': 'TV', 'JSD': 'Jensen-Shannon Divergence'}
# metrics = ['TV']
game_stages = ['early', 'intermediate', 'late']

# Define agents with desired order
agents = [#'split-human',
          'ours', 'expert', 'random']

agg_scores_alpha= {}
individ_move_scores_alpha = {}

split_halfs = []

metric ='TV'# 'JSD'
# metric= 'JSD'

# get number of matches -- just pull one agent [same for all]
agent = 'ours'
agent_df = metric_df[metric_df['agent'] == agent]
arenas = sorted(list(set(agent_df['arena'])))
print("num arenas: ", len(arenas))
n_matches = 0
for arena in arenas: 
    arena_df = agent_df.loc[agent_df.arena == arena]
    #arena_games = sorted(list(arena_df['game'])) 
    arena_games = list(set(arena_df['game']))
    print(len(arena_games))
    n_matches += len(arena_games)

print("num matches: ", n_matches)
    
individ_move_scores_per_alpha_per_game = {}
individ_move_scores_per_stage = {}
individ_move_scores_per_stage_per_game = {}
individ_move_scores_per_alpha_per_agent = {}
    
for agent_idx, agent in enumerate(viz_agents): 
    print(agent)
    agent_df = metric_df[metric_df['agent'] == agent]#[metrics[0]]
    collapsed_scores = []
    alpha_vals = sorted(list(set(agent_df['alpha'])))
    full_alpha_vals = alpha_vals # save
    print(full_alpha_vals)
    match_scores = np.zeros(n_matches)
    individ_move_scores = np.zeros(n_matches * len(game_stages))
    
    individ_per_game_stage = {game_stage: np.zeros(n_matches) for game_stage in game_stages}
    
    # get max num moves -- todo: better code!!
    max_num_moves = 0
    for arena in arenas: 
        arena_df = agent_df.loc[agent_df.arena == arena]
        arena_games = sorted(list(set(arena_df['game'])))
        for game in arena_games: 
            for stage in game_stages: 
                max_num_moves += 1

    individ_move_scores_per_alpha = np.zeros([len(alpha_vals), max_num_moves])
    
    
    for alpha_idx, alpha in enumerate(alpha_vals): 
        curr_move_idx = 0
        match_idx = 0
        move_idx2game = {}
        match_idx2game = {}
        for arena in arenas: 
            arena_df = agent_df.loc[agent_df.arena == arena]
            arena_games = sorted(list(set(arena_df['game'])))
            for game in arena_games: 
                for stage in game_stages: 
                    #print(len(arena_df.loc[(arena_df.stage == stage) & (arena_df.alpha == alpha)]))
                    # if len(arena_df.loc[(arena_df.stage == stage)]) > 1: print(arena, len(arena_df.loc[(arena_df.stage == stage)]))
                    stage_df = arena_df.loc[(arena_df.stage == stage) & (arena_df.game == game) & (arena_df.alpha == alpha)].iloc[0] #& (arena_df.alpha == alpha)].iloc[0]
                    #print(stage_df.columns)
                    score = stage_df[metric]
                    match_scores[match_idx] += score
                    individ_move_scores[curr_move_idx] += score
                    move_idx2game[curr_move_idx] = game
                    #print("current move: ", curr_move_idx)
                    individ_move_scores_per_alpha[alpha_idx, curr_move_idx] = score
                    curr_move_idx += 1
                    
                    match_idx2game[match_idx] = game
                    
                    individ_per_game_stage[stage][match_idx] += score
                    
                    
                    
                    
                match_idx += 1

    match_scores /= (len(alpha_vals) * 3)
    individ_move_scores /= len(alpha_vals)
    individ_per_game_stage = {stage: vals/len(alpha_vals) for stage, vals in individ_per_game_stage.items()}
    agg_scores_alpha[agent] = match_scores #collapsed_scores
    individ_move_scores_alpha[agent] = individ_move_scores
    individ_move_scores_per_stage[agent] = individ_per_game_stage
    
    individ_move_scores_per_alpha_per_game[agent] = {}
    for move_idx in range(curr_move_idx):#-1): 
        game = move_idx2game[move_idx]
        individ_move_scores_per_alpha_per_game[agent].setdefault(game, [])
        #print(move_idx, curr_move_idx)
        individ_move_scores_per_alpha_per_game[agent][game].append(individ_move_scores[move_idx])
        
    individ_move_scores_per_stage_per_game[agent] = {}
    for game_stage in game_stages: 
        for idx in range(match_idx): 
            game = match_idx2game[idx]
            individ_move_scores_per_stage_per_game[agent].setdefault(game_stage, {})
            individ_move_scores_per_stage_per_game[agent][game_stage].setdefault(game, [])
            individ_move_scores_per_stage_per_game[agent][game_stage][game].append(individ_per_game_stage[game_stage][idx])
            
    individ_move_scores_per_alpha_per_agent[agent] = individ_move_scores_per_alpha
    
# get split half preds ordered
agent = 'split-human'
agent_df = metric_df[metric_df['agent'] == agent]
alpha_vals = [0.999]
match_scores = np.zeros(n_matches)
individ_move_scores = np.zeros(n_matches * len(game_stages))
individ_per_game_stage = {game_stage: np.zeros(n_matches) for game_stage in game_stages}
for alpha_idx, alpha in enumerate(alpha_vals): 
    curr_move_idx = 0
    match_idx = 0
    move_idx2game = {}
    match_idx2game = {}
    for arena in arenas: 
        arena_df = agent_df.loc[agent_df.arena == arena]
        arena_games = sorted(list(set(arena_df['game'])))
        for game in arena_games: 
            for stage in game_stages: 
                #print(len(arena_df.loc[(arena_df.stage == stage) & (arena_df.alpha == alpha)]))
                # if len(arena_df.loc[(arena_df.stage == stage)]) > 1: print(arena, len(arena_df.loc[(arena_df.stage == stage)]))
                stage_df = arena_df.loc[(arena_df.stage == stage) & (arena_df.game == game)].iloc[0] #& (arena_df.alpha == alpha)].iloc[0]
                score = stage_df[metric]
                match_scores[match_idx] += score
                individ_move_scores[curr_move_idx] += score
                move_idx2game[curr_move_idx] = game
                curr_move_idx += 1
                match_idx2game[match_idx] = game
                individ_per_game_stage[stage][match_idx] += score
                #individ_move_scores_per_alpha[alpha_idx, curr_move_idx] = score
            match_idx += 1
match_scores /= (len(alpha_vals) * 3)
individ_move_scores /= len(alpha_vals)
agg_scores_alpha[agent] = match_scores #collapsed_scores
individ_move_scores_alpha[agent] = individ_move_scores
individ_move_scores_per_alpha_per_game[agent] = {}
individ_per_game_stage = {stage: vals/len(alpha_vals) for stage, vals in individ_per_game_stage.items()}
individ_move_scores_per_stage[agent] = individ_per_game_stage

#individ_move_scores_per_agent_per_alpha[agent] = individ_move_scores_per_alpha

for move_idx in range(curr_move_idx): #len(individ_move_scores)): #range(curr_move_idx-1): 
    game = move_idx2game[move_idx]
    individ_move_scores_per_alpha_per_game[agent].setdefault(game, [])
    individ_move_scores_per_alpha_per_game[agent][game].append(individ_move_scores[move_idx])
    
individ_move_scores_per_stage_per_game[agent] = {}
for game_stage in game_stages: 
    for idx in range(match_idx): 
        game = match_idx2game[idx]
        individ_move_scores_per_stage_per_game[agent].setdefault(game_stage, {})
        individ_move_scores_per_stage_per_game[agent][game_stage].setdefault(game, [])
        individ_move_scores_per_stage_per_game[agent][game_stage][game].append(individ_per_game_stage[game_stage][idx])
    
alpha_vals = full_alpha_vals

In [ ]:

viz_agents = ['split-human', 'ours', 'ours-depth3', 'expert', 'random']
game_stages = ['early', 'intermediate', 'late']
n_boot = 1000

fig, axes = plt.subplots(1, len(game_stages), figsize=(20, 6), sharey=True)
for stage_idx, stage in enumerate(game_stages):
    ax = axes[stage_idx]
    agent_stats = {}
    for agent in viz_agents:
        scores = np.array(list(individ_move_scores_per_stage[agent][stage]))
        m, lo, hi = analysis_utils.bootstrap_mean_lo_hi(scores, n_boot=n_boot)
        agent_stats[agent] = {'mean': m, 'lower': lo, 'upper': hi}

    labels = [model2name[a].replace(" ", "\n") for a in viz_agents]
    means = [agent_stats[a]['mean'] for a in viz_agents]
    colors = [agent2color[a] for a in viz_agents]
    yerr = [
        [agent_stats[a]['mean'] - agent_stats[a]['lower'] for a in viz_agents],
        [agent_stats[a]['upper'] - agent_stats[a]['mean'] for a in viz_agents],
    ]
    ax.bar(labels, means, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.errorbar(labels, means, yerr=yerr, fmt='none', ecolor='black',
                capsize=5, linewidth=1.5)
    ax.set_ylabel("Total Variation Distance", fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.set_title(stage.capitalize(), fontsize=24)

    for agent in viz_agents:
        s = agent_stats[agent]
        print(f"{stage.capitalize()} {agent.capitalize()}: "
              f"mean={s['mean']:.4f}, 95% CI=[{s['lower']:.4f}, {s['upper']:.4f}]")

plt.tight_layout()
plt.savefig('tvd_stages.pdf', dpi=400)
plt.show()


In [ ]:

sub_vizagents = ['ours', 'expert', 'random']
model2name['split-human'] = 'Split-Halve\nHuman'
agent2color['split-human'] = 'pink'
n_boot = 1000

metric = 'TVD'

agent_stats = {}
for agent in sub_vizagents:
    vals = [np.mean(v) for v in individ_move_scores_per_alpha_per_game[agent].values()]
    print(len(vals))
    mean, lower, upper = analysis_utils.bootstrap_column_subsample(vals, n_boot=n_boot)
    agent_stats[agent] = {'mean': float(mean[0]),
                          'lower': float(lower[0]),
                          'upper': float(upper[0])}

# Split-human reference (regular 1-D bootstrap, not column-subsample)
sh_mean, sh_lo, sh_hi = analysis_utils.bootstrap_mean_lo_hi(
    agg_scores_alpha['split-human'], n_boot=n_boot
)
split_human_stats = {'mean': sh_mean, 'lower': sh_lo, 'upper': sh_hi}

ylabel = ("Jensen-Shannon Divergence" if metric == 'JSD'
          else "Total Variation Distance")
analysis_utils.plot_agent_bar_with_ci(
    agent_stats, sub_vizagents, agent2color, model2name,
    ylabel=ylabel,
    savepath=f'final_figures/watch/aggregate_{metric}_lines.svg',
    hline=split_human_stats['mean'],
    hline_band=(split_human_stats['lower'], split_human_stats['upper']),
)

save_viz_data = pd.DataFrame({
    'agents': [model2name[a].replace(" ", "\n") for a in sub_vizagents] + ['split-human-avg'],
    'means':  [agent_stats[a]['mean'] for a in sub_vizagents] + [[split_human_stats['mean']]],
    'err_low':  [agent_stats[a]['mean'] - agent_stats[a]['lower'] for a in sub_vizagents]
              + [[split_human_stats['lower']]],
    'err_high': [agent_stats[a]['upper'] - agent_stats[a]['mean'] for a in sub_vizagents]
              + [[split_human_stats['upper']]],
})
save_viz_data.to_csv(fig_data_file_pth + f'watch_main_{metric}.csv')


In [ ]:

comparisons = [('ours', 'expert'), ('ours', 'random')]
comp_diffs = {}
comp_stats = {}

for comp in comparisons:
    agent1, agent2 = comp
    
    diffs2 = [a1-a2 for a1, a2 in zip(individ_move_scores_alpha[agent1], individ_move_scores_alpha[agent2])]
    comp_diffs[tuple(comp)] = diffs2
    
    t_stat, p_value = stats.ttest_rel(individ_move_scores_alpha[agent1], individ_move_scores_alpha[agent2], alternative='less')
    
    # Store results
    comp_stats[tuple(comp)] = {
        'mean_diff': np.mean(diffs2),
        't_statistic': t_stat,
        'p_value': p_value,
        'significant': p_value < 0.05
    }

    print(np.median(diffs2))
print(individ_move_scores_alpha['ours'].shape, len(diffs2))

for comp, stats_dict in comp_stats.items():
    print(f"Comparison: {comp[0]} vs {comp[1]}")
    print(f"Mean difference: {stats_dict['mean_diff']:.10f}")
    print(f"t-statistic: {stats_dict['t_statistic']:.4f}")
    print(f"p-value: {stats_dict['p_value']}")#:.4f}")
    print(f"Statistically significant (p<0.05): {stats_dict['significant']}")
    print("-" * 40)

plt.figure(figsize=(12, 5))
for i, (comp, diffs2) in enumerate(comp_diffs.items()):
    plt.subplot(1, len(comparisons), i+1)
    plt.hist(diffs2, bins=10, alpha=0.7)
    plt.axvline(x=0, color='red', linestyle='--')
    plt.title(f"{comp[0]} vs {comp[1]}\nMean diff: {np.mean(diffs2):.4f}")
    plt.xlabel("Difference (Group1 - Group2)")
    plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:

viz_agents = ['ours', 'ours-depth3', 'expert', 'random']

metric_df = pd.read_csv(f"{main_dir}watch/metrics_agg.csv")
metrics = ['TV', 'JSD']

for metric in metrics: 
    agents = ['split-human', 'ours', 'expert', 'random']
    all_games = sorted(metric_df['game'].unique())  # All games
    agent2color['split-human'] = 'pink'
    # Calculate average TVD for split-human for ordering
    game_tvd_order = {}
    for game in all_games:
        human_data = metric_df[(metric_df['agent'] == 'split-human') & (metric_df['game'] == game)]
        arenas = sorted(list(set(human_data['arena'])))
        stage_vals = []
        for arena in arenas:
            for stage in game_stages:
                stage_val = human_data[(human_data['arena'] == arena) & (human_data['stage'] == stage)].iloc[0][metric]
                stage_vals.append(stage_val)
        game_tvd_order[game] = np.mean(stage_vals)

    sorted_games = sorted(all_games, key=lambda g: game_tvd_order[g])

    plt.figure(figsize=(20, 8))
    x = np.arange(len(sorted_games))
    width = 0.2
    offsets = [-1.5*width, -0.5*width, 0.5*width, 1.5*width]

    for i, agent in enumerate(agents):
        means = []
        cis = []  

        for game in sorted_games:
            game_data = metric_df[(metric_df['agent'] == agent) & (metric_df['game'] == game)]
            
            if agent == 'split-human':
                arenas = sorted(list(set(game_data['arena'])))
                stage_vals = []
                for arena in arenas:
                    for stage in game_stages:
                        stage_val = game_data[(game_data['arena'] == arena) & (game_data['stage'] == stage)].iloc[0][metric]
                        stage_vals.append(stage_val)
                vals = np.array(stage_vals)
            else:
                alpha_vals = sorted(list(set(game_data['alpha'])))
                arenas = sorted(list(set(game_data['arena'])))
                stage_vals = np.zeros(len(arenas) * len(game_stages))
                for alpha in alpha_vals:
                    move_idx = 0
                    for arena in arenas:
                        for stage_idx, stage in enumerate(game_stages):
                            val = game_data.loc[(game_data.alpha == alpha) & (game_data.stage == stage) & (game_data.arena == arena)][metric].iloc[0]
                            stage_vals[move_idx] += val
                            move_idx += 1
                stage_vals /= len(alpha_vals)
                vals = np.array(stage_vals)
            
            mean_value = np.mean(vals)
            ci_low = np.std(vals)
            ci_high = np.std(vals)

            means.append(mean_value)
            cis.append([ci_low, ci_high])

        pos = x + offsets[i]
        errors = np.array(cis).T  # shape (2, N)

        plt.bar(pos, means, width, label=model2name[agent],
                color=agent2color[agent], alpha=0.8, yerr=errors, capsize=3)

    # Format x-axis with game names
    game_labels = []
    importlib.reload(analysis_utils)
    for game in sorted_games:
        game = analysis_utils.tidy_game_code(game)
        game_labels.append(game)

    plt.xticks(x, game_labels, rotation=45, fontsize=26, ha='right')

    # Add y-label
    metric_label = 'Total Variation Distance' if metric == 'TV' else metric2viz[metric]
    plt.ylabel(metric_label, fontsize=28)

    plt.grid(False)
    plt.subplots_adjust(bottom=0.25)
    plt.tight_layout()
    plt.savefig(f"{main_save}watch/game_{metric}.svg", dpi=400)


In [ ]:
metric_df = pd.read_csv(f"{main_dir}watch/logprob_agg.csv")

metric2viz = {'log_pmove': 'Aggregate Log Prob'}
agent2color['should'] = 'pink'
metrics = ['log_pmove']

model2name['should'] = 'Human Watchers'
agents = ['should', 'ours', 'expert', 'random']

In [ ]:



fig, axes = plt.subplots(len(metrics), 1, figsize=(5.5, 6))

for i, metric in enumerate(metrics):
    ax = axes if len(metrics) == 1 else axes[i]
    mean_scores = []
    ci_scores = []

    for agent in agents:
        values = metric_df[metric_df['agent'] == agent][metric].dropna().values
        if len(values) == 0:
            mean_scores.append(np.nan)
            ci_scores.append(0)
        else:
            mean, ci = analysis_utils.bootstrapped_mean_ci(values, n_boot=1000, ci=95)
            mean_scores.append(-mean)
            ci_scores.append(ci)

    x_positions = np.arange(len(agents)) * 0.8

    bars = ax.bar(
        x_positions,
        mean_scores,
        width=0.6,
        alpha=0.7,
        color=[agent2color[agent] for agent in agents],
        edgecolor='black',
        linewidth=1.5
    )

    ax.errorbar(
        x_positions,
        mean_scores,
        yerr=ci_scores,
        fmt='none',
        c='black',
        capsize=10,
        linewidth=2.0,
        elinewidth=2.0,
        capthick=1.5
    )
    # X labels
    ax.set_xticks(x_positions, [model2name[agent].replace(" ", "\n") for agent in agents])
    ax.set_ylabel(metric2viz[metric], fontsize=20)
    ax.tick_params(axis='both', labelsize=20)
    #ax.set_xlabel('Agg NLL', fontsize=20)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_xlim(x_positions.min() - 0.4, x_positions.max() + 0.4)
    ax.margins(x=0.01)
    ax.grid(False)

plt.tight_layout()
plt.savefig(f"{main_save}/watch/agg_logprob.svg", dpi=400)

In [ ]:


metric = metrics[0]
agents = ['should', 'ours', 'expert', 'random']

all_games = sorted(metric_df['game'].unique())  
metric = 'log_pmove'

game_stages = sorted(metric_df['stage'].unique())

# Calculate the average NLL for split-human for each game to determine order
game_nll_order = {}
for game in all_games:
    human_data = metric_df[(metric_df['agent'] == 'should') & (metric_df['game'] == game)]
    arenas = sorted(list(set(human_data['arena'])))
    stage_vals = []
    for arena in arenas:
        for stage in game_stages:
            sub = human_data[(human_data['arena'] == arena) & (human_data['stage'] == stage)]
            if not sub.empty:
                stage_vals.append(sub.iloc[0][metric])
    game_nll_order[game] = np.mean(stage_vals) if len(stage_vals) > 0 else float('inf')

# Sort games by increasing split-human NLL
sorted_games = sorted(all_games, key=lambda g: game_nll_order[g], reverse=True)

plt.figure(figsize=(20, 8))
x = np.arange(len(sorted_games))
width = 0.2
offsets = [-1.5*width, -0.5*width, 0.5*width, 1.5*width]
save_viz_data = {'source': [], 'means': [], 'errs': []}
for i, agent in enumerate(agents):
    means = []
    errors = []
    for game in sorted_games:
        game_data = metric_df[(metric_df['agent'] == agent) & (metric_df['game'] == game)]
        arenas = sorted(list(set(game_data['arena'])))
        stage_vals = []
        for arena in arenas:
            for stage in game_stages:
                sub = game_data[(game_data['arena'] == arena) & (game_data['stage'] == stage)]
                if not sub.empty:
                    stage_vals.append(-sub.iloc[0][metric])
        if stage_vals:
            mean_value = np.mean(stage_vals)
            std_value = np.std(stage_vals) 
        else:
            mean_value = 0
            std_value = 0
        means.append(mean_value)
        errors.append(std_value)
    save_viz_data['source'].append(agent)
    save_viz_data['means'].append(means)
    save_viz_data['errs'].append(errors)

    pos = x + offsets[i]
    plt.bar(pos, means, width, label=model2name[agent],
            color=agent2color[agent], alpha=0.8, yerr=errors, capsize=3)

game_labels = []
for game in sorted_games:
    n_rows, n_cols, win_conds = game.split("*")
    n_rows = int(float(n_rows))
    n_cols = int(float(n_cols))
    game_label = analysis_utils.tidy_game_code(game)
    game_labels.append(game_label)


plt.xticks(x, game_labels, rotation=45, fontsize=20, ha='right')


save_viz_data = pd.DataFrame(save_viz_data)
save_viz_data.to_csv(fig_data_file_pth + 'per_game_watch.csv')

plt.ylabel('Aggregate Log Prob', fontsize=26)
plt.grid(False)
plt.subplots_adjust(bottom=0.25)
plt.tight_layout()
plt.savefig(f"{main_save}watch/game_agglog.svg", dpi=400)
